# Load Library

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset

In [2]:
# dataset source: https://www.kaggle.com/datasets/hellbuoy/online-retail-customer-clustering

import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "hellbuoy/online-retail-customer-clustering",
    "OnlineRetail.csv",
    pandas_kwargs={"encoding": "ISO-8859-1"},
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [4]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

# Data Cleaning

## CustomerID
I can drop data that contain missing value in 'CustomerID' column. Becase in this case, i will only make a clustering use K-mean and Hierarchical. And I will change data type from float to integer to str. Why integer first? Because, if i change directly from float to str, there will be commas in that data. So i wanna remove that commas use astype(int), and then change it again to str.

In [5]:
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)
df['CustomerID'] = df['CustomerID'].apply(str)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    406829 non-null  object 
 1   StockCode    406829 non-null  object 
 2   Description  406829 non-null  object 
 3   Quantity     406829 non-null  int64  
 4   InvoiceDate  406829 non-null  object 
 5   UnitPrice    406829 non-null  float64
 6   CustomerID   406829 non-null  object 
 7   Country      406829 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 27.9+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom


In [6]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

## InvoiceDate
I will separate date and timestamp in this column. Date in the 'InvoiceDate' column and timestamp in the 'InvoiceTime'.


In [7]:
df[['InvoiceDate', 'InvoiceTime']] = df['InvoiceDate'].str.split(' ', expand=True)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    406829 non-null  object 
 1   StockCode    406829 non-null  object 
 2   Description  406829 non-null  object 
 3   Quantity     406829 non-null  int64  
 4   InvoiceDate  406829 non-null  object 
 5   UnitPrice    406829 non-null  float64
 6   CustomerID   406829 non-null  object 
 7   Country      406829 non-null  object 
 8   InvoiceTime  406829 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 31.0+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceTime
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010,2.55,17850,United Kingdom,08:26
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010,3.39,17850,United Kingdom,08:26
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010,2.75,17850,United Kingdom,08:26
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010,3.39,17850,United Kingdom,08:26
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010,3.39,17850,United Kingdom,08:26


## Quantity

In [8]:
df['Quantity'].value_counts()

Quantity
 1        73314
 12       60033
 2        58003
 6        37688
 4        32183
          ...  
 234          1
 1404         1
 698          1
 80995        1
-80995        1
Name: count, Length: 436, dtype: int64

In [9]:
df[df['Quantity'] < 1]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceTime
141,C536379,D,Discount,-1,01-12-2010,27.50,14527,United Kingdom,09:41
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,01-12-2010,4.65,15311,United Kingdom,09:49
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,01-12-2010,1.65,17548,United Kingdom,10:24
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,01-12-2010,0.29,17548,United Kingdom,10:24
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,01-12-2010,0.29,17548,United Kingdom,10:24
...,...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,09-12-2011,0.83,14397,United Kingdom,09:57
541541,C581499,M,Manual,-1,09-12-2011,224.69,15498,United Kingdom,10:28
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,09-12-2011,10.95,15311,United Kingdom,11:57
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,09-12-2011,1.25,17315,United Kingdom,11:58


So, in 'Quantity' column, there are anomaly value (< 1). I will drop that, because that doesn't make sense if we buy something and the quantity is under 1.

In [10]:
df = df[df['Quantity'] >= 1]

df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    397924 non-null  object 
 1   StockCode    397924 non-null  object 
 2   Description  397924 non-null  object 
 3   Quantity     397924 non-null  int64  
 4   InvoiceDate  397924 non-null  object 
 5   UnitPrice    397924 non-null  float64
 6   CustomerID   397924 non-null  object 
 7   Country      397924 non-null  object 
 8   InvoiceTime  397924 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 30.4+ MB


,Quantity,UnitPrice
count,397924.000000,397924.000000
mean,13.021823,3.116174
std,180.420210,22.096788
min,1.000000,0.000000
25%,2.000000,1.250000
50%,6.000000,1.950000
75%,12.000000,3.750000
max,80995.000000,8142.750000


In [11]:
df['Quantity'].value_counts()

Quantity
1        73314
12       60033
2        58003
6        37688
4        32183
         ...  
59           1
388          1
1404         1
698          1
80995        1
Name: count, Length: 302, dtype: int64

## UnitPrice

In [12]:
df[df['UnitPrice'] <= 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceTime
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,05-12-2010,0.0,12647,Germany,14:02
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,16-12-2010,0.0,16560,United Kingdom,14:36
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,21-12-2010,0.0,14911,EIRE,13:45
47068,540372,22090,PAPER BUNTING RETROSPOT,24,06-01-2011,0.0,13081,United Kingdom,16:41
47070,540372,22553,PLASTERS IN TIN SKULLS,24,06-01-2011,0.0,13081,United Kingdom,16:41
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,13-01-2011,0.0,15107,United Kingdom,15:10
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,10-02-2011,0.0,17560,United Kingdom,13:08
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,23-03-2011,0.0,13239,United Kingdom,10:25
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,30-03-2011,0.0,13113,United Kingdom,12:45
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,04-04-2011,0.0,14410,United Kingdom,14:42
